In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, input):
        lstm_out, _ = self.lstm(input)
        output = self.fc(lstm_out[:, -1, :])  # taking only the last time step output
        return output

class ChannelPrediction: 
    def __init__(self, H_b, t, sequence_length, BS_NO, USER_NO): # default sequence length = 3
        self.H_b = H_b #  BS_NO * USER_NO
        self.t = t
        self.sequence_length = sequence_length
        self.BS_NO = BS_NO
        self.USER_NO = USER_NO

    def train_model(self):
        model = LSTMModel(input_size=self.BS_NO * self.USER_NO,
                          hidden_size=50,
                          output_size=self.BS_NO * self.USER_NO)
        
        loss_function = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        # Pad the input data with zeros for initial time steps
        padded_data = np.zeros((self.sequence_length, self.BS_NO, self.USER_NO))
        padded_data[-self.t:] = self.H_b[:self.t]

        for epoch in range(200):
            for t in range(self.sequence_length, self.t):
                data = torch.tensor(padded_data[t-self.sequence_length:t], dtype=torch.float32).unsqueeze(0)
                labels = torch.tensor(self.H_b[t], dtype=torch.float32).unsqueeze(0)

                optimizer.zero_grad()
                output = model(data)
                loss = loss_function(output, labels)
                loss.backward()
                optimizer.step()

    
    def predict_next_channel(self):
        model = LSTMModel(input_size=self.BS_NO * self.USER_NO,
                          hidden_size=50,
                          output_size=self.BS_NO * self.USER_NO)
        # Load trained model weights
        #model.load_state_dict(torch.load('lstm_model_weights.pth'))

        next_channel = model(torch.tensor(self.H_b[self.t-self.sequence_length+1:self.t+1], dtype=torch.float32).unsqueeze(0))
        return next_channel.detach().numpy()


T=10
BS_NO=2
USER_NO=5
sequence_length=3

for t in range(T):
    H_b = np.random.rand(BS_NO, USER_NO)
    channel_prediction = ChannelPrediction(H_b, t, sequence_length, BS_NO, USER_NO)
    channel_prediction.train_model()

    next_channel = channel_prediction.predict_next_channel()
    print(next_channel)

ValueError: could not broadcast input array from shape (0,5) into shape (3,2,5)